# ORESTAR fundraising explorer (2024)
 measures:

- total amount;
- total record count;
- record count in each size bin;
- record share in each size bin.

Then we do two things:

1. run a correlation race for all of those X variables;
2. choose one X variable and draw one OLS regression plot.




## 1. Setup

This notebook is self-contained for the analysis step: it reads the
candidate-level profile table already built for this source and the
ballot-support table.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from IPython.display import display
from matplotlib.ticker import (
    PercentFormatter,
    StrMethodFormatter,
)


# ---------------------------------------------------------------
# Find the repository root
# ---------------------------------------------------------------

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():

    ROOT = cwd

elif (cwd.parent / "pyproject.toml").exists():

    ROOT = cwd.parent

else:

    raise FileNotFoundError(
        "Could not find pyproject.toml in this folder or its parent."
    )


print(
    "ROOT:",
    ROOT,
)


ROOT: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis




`RACE_DISTRICT` controls the correlation table.

`PLOT_DISTRICT` controls the regression plot.

Use `"all"` for all four districts together, or use `1`, `2`, `3`, or `4`.

`X_METRIC` is the finance variable you want on the x-axis.

`Y_METRIC` can be `"mentions"` or `"first_place_votes"`.


In [2]:
YEAR = 2024

# Correlation race sample
RACE_DISTRICT = "all"

# Regression plot sample
PLOT_DISTRICT = 4

# Change this line to play with another finance metric
X_METRIC = "total_amount"

# Options: "mentions", "first_place_votes"
Y_METRIC = "mentions"

SHOW_NAMES = True


In [3]:
SUPPORT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "ballot_support"
    / str(YEAR)
    / f"candidate_ballot_support_{YEAR}.csv"
)


PROFILE_PATH = (
    ROOT
    / "data/processed/fundraising/2024/city_council/orestar_candidate_fundraising_profiles_wide.csv"
)


for path in [
    SUPPORT_PATH,
    PROFILE_PATH,
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing required file: {path}"
        )


support = pd.read_csv(
    SUPPORT_PATH,
    low_memory=False,
)


profile = pd.read_csv(
    PROFILE_PATH,
    low_memory=False,
)


# Keep only profiles linked to an official candidate.
profile = profile[
    profile["candidate_key"].notna()
].copy()


# A duplicated candidate key would make the merge ambiguous.
if profile["candidate_key"].duplicated().any():

    duplicates = profile[
        profile["candidate_key"].duplicated(
            keep=False
        )
    ].copy()

    display(
        duplicates[
            [
                column
                for column in [
                    "district",
                    "candidate_key",
                    "canonical_candidate",
                    "candidate",
                ]
                if column in duplicates.columns
            ]
        ]
    )

    raise ValueError(
        "Duplicate candidate_key values found in the finance profile."
    )


print(
    "Ballot-support candidates:",
    len(support),
)

print(
    "ORESTAR fundraising profiles:",
    len(profile),
)


FileNotFoundError: Missing required file: /Users/marcy/mggg/projects/Portland/portland-fundraising-support-analysis/data/processed/fundraising/2024/city_council/orestar_candidate_fundraising_profiles_wide.csv

## 4. The X variables

These are the only finance variables in this small notebook.


In [ ]:
BINS = [
    "micro",
    "small",
    "medium",
    "large",
    "mega",
]


METRICS = [
    "total_amount",
    "total_contribution_count",
]


for bin_name in BINS:

    METRICS.append(
        f"contribution_count_{bin_name}"
    )

    METRICS.append(
        f"contribution_share_{bin_name}"
    )


METRIC_LABELS = {
    "total_amount":
        "Total ORESTAR fundraising ($)",

    "total_contribution_count":
        "ORESTAR contribution records",
}


for bin_name in BINS:

    nice_bin = bin_name.title()

    METRIC_LABELS[
        f"contribution_count_{bin_name}"
    ] = (
        f"{nice_bin} contribution records"
    )

    METRIC_LABELS[
        f"contribution_share_{bin_name}"
    ] = (
        f"{nice_bin} share of contribution records"
    )


missing_metrics = [
    metric
    for metric in METRICS
    if metric not in profile.columns
]


if missing_metrics:

    raise ValueError(
        "These expected metrics are missing from the profile table: "
        + str(missing_metrics)
    )


print("Available X metrics:")

for metric in METRICS:
    print(
        "-",
        metric,
        "→",
        METRIC_LABELS[metric],
    )


## 5. Merge finance with ballot support

Ballot support is the main candidate table. Finance variables are added
when a matching `candidate_key` is available.


In [ ]:
keep_columns = [
    "year",
    "district",
    "candidate_key",
] + METRICS


analysis = (
    support
    .merge(
        profile[
            keep_columns
        ],
        on=[
            "year",
            "district",
            "candidate_key",
        ],
        how="left",
        validate="one_to_one",
    )
)


print(
    "Candidates:",
    len(analysis),
)

print(
    "Candidates with finance data:",
    analysis[
        METRICS[0]
    ].notna().sum(),
)


display(
    analysis[
        [
            "district",
            "canonical_candidate",
            "mentions",
            "first_place_votes",
        ]
        + METRICS
    ]
    .head(10)
)


## 6. Small correlation function

For every X variable we calculate:

- `n`
- Pearson `r`
- Spearman `r`

We do this for both ballot outcomes.


In [ ]:
def calculate_correlations(
    data,
    metrics,
    outcomes,
):

    rows = []


    for outcome in outcomes:

        for metric in metrics:

            pair = (
                data[
                    [
                        metric,
                        outcome,
                    ]
                ]
                .dropna()
            )


            if (
                len(pair) < 3
                or pair[metric].nunique() < 2
                or pair[outcome].nunique() < 2
            ):

                continue


            rows.append(
                {
                    "outcome":
                        outcome,

                    "metric":
                        metric,

                    "n":
                        len(pair),

                    "pearson_r":
                        pair[metric].corr(
                            pair[outcome],
                            method="pearson",
                        ),

                    "spearman_r":
                        pair[metric].corr(
                            pair[outcome],
                            method="spearman",
                        ),
                }
            )


    return pd.DataFrame(
        rows
    )


## 7. Correlation race

The race uses `RACE_DISTRICT`.

It ranks all of our X variables by the absolute value of Pearson `r`.


In [ ]:
if RACE_DISTRICT == "all":

    race_data = analysis.copy()

else:

    race_data = analysis[
        analysis["district"].eq(
            RACE_DISTRICT
        )
    ].copy()


correlations = calculate_correlations(
    race_data,
    metrics=METRICS,
    outcomes=[
        "mentions",
        "first_place_votes",
    ],
)


correlations["label"] = (
    correlations["metric"]
    .map(
        METRIC_LABELS
    )
)


for outcome in [
    "mentions",
    "first_place_votes",
]:

    race = (
        correlations[
            correlations["outcome"].eq(
                outcome
            )
        ]
        .assign(
            abs_pearson=lambda data:
                data["pearson_r"].abs()
        )
        .sort_values(
            "abs_pearson",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    race.insert(
        0,
        "rank",
        np.arange(
            1,
            len(race) + 1,
        ),
    )


    print()
    print(
        f"CORRELATION RACE — {outcome}"
    )

    print(
        "District:",
        RACE_DISTRICT,
    )


    display(
        race[
            [
                "rank",
                "metric",
                "label",
                "n",
                "pearson_r",
                "spearman_r",
            ]
        ]
        .round(
            {
                "pearson_r": 3,
                "spearman_r": 3,
            }
        )
    )


## 8. Regression plot

This is the meeting plot.

Change only:

```python
X_METRIC = "..."
PLOT_DISTRICT = ...
Y_METRIC = "..."
```

The points are not grouped or specially colored.

Candidate labels are staggered around the points to reduce overlap.
If `"all"` is too crowded, choose one district.


In [ ]:
def add_candidate_labels(
    ax,
    data,
    x_column,
    y_column,
):
    # A small set of different label positions.
    # We cycle through them so labels do not all sit
    # in exactly the same place relative to the point.

    offsets = [
        (6, 8),
        (6, -12),
        (-6, 8),
        (-6, -12),
        (12, 16),
        (12, -20),
        (-12, 16),
        (-12, -20),
        (18, 2),
        (-18, 2),
        (0, 22),
        (0, -26),
    ]


    ordered = (
        data
        .sort_values(
            [
                x_column,
                y_column,
            ]
        )
        .reset_index(
            drop=True
        )
    )


    for i, row in ordered.iterrows():

        x_offset, y_offset = (
            offsets[
                i % len(offsets)
            ]
        )


        ax.annotate(
            str(
                row[
                    "canonical_candidate"
                ]
            ),
            (
                row[x_column],
                row[y_column],
            ),
            xytext=(
                x_offset,
                y_offset,
            ),
            textcoords="offset points",
            fontsize=8,
            ha=(
                "left"
                if x_offset >= 0
                else "right"
            ),
            va="center",
        )


In [ ]:
def plot_simple_regression(
    data,
    x_column,
    y_column,
    district="all",
    show_names=True,
):

    if x_column not in METRICS:

        raise ValueError(
            "X_METRIC is not in METRICS. "
            "Look at the printed metric list above."
        )


    if y_column not in [
        "mentions",
        "first_place_votes",
    ]:

        raise ValueError(
            "Y_METRIC must be 'mentions' "
            "or 'first_place_votes'."
        )


    if district == "all":

        plot_data = data.copy()

    else:

        plot_data = data[
            data["district"].eq(
                district
            )
        ].copy()


    plot_data = (
        plot_data[
            [
                "canonical_candidate",
                x_column,
                y_column,
            ]
        ]
        .dropna()
        .copy()
    )


    if len(plot_data) < 3:

        raise ValueError(
            "Not enough complete observations for this regression."
        )


    # -----------------------------------------------------------
    # OLS
    # -----------------------------------------------------------

    x = plot_data[
        x_column
    ].astype(float).to_numpy()

    y = plot_data[
        y_column
    ].astype(float).to_numpy()


    X = sm.add_constant(
        x
    )

    model = sm.OLS(
        y,
        X,
    ).fit()


    x_plot = np.linspace(
        x.min(),
        x.max(),
        300,
    )


    X_plot = sm.add_constant(
        x_plot
    )


    prediction = (
        model
        .get_prediction(
            X_plot
        )
        .summary_frame(
            alpha=0.05
        )
    )


    # -----------------------------------------------------------
    # Plot
    # -----------------------------------------------------------

    figure_height = max(
        7,
        6 + len(plot_data) * 0.08,
    )


    fig, ax = plt.subplots(
        figsize=(
            12,
            figure_height,
        )
    )


    ax.scatter(
        x,
        y,
        alpha=0.75,
    )


    ax.plot(
        x_plot,
        prediction[
            "mean"
        ],
        label="OLS fit",
    )


    ax.fill_between(
        x_plot,
        prediction[
            "mean_ci_lower"
        ],
        prediction[
            "mean_ci_upper"
        ],
        alpha=0.15,
        label="95% CI",
    )


    if show_names:

        add_candidate_labels(
            ax,
            plot_data,
            x_column,
            y_column,
        )


    # -----------------------------------------------------------
    # Friendly axis formatting
    # -----------------------------------------------------------

    ax.set_xlabel(
        METRIC_LABELS[
            x_column
        ]
    )

    ax.set_ylabel(
        (
            "Ballot mentions"
            if y_column == "mentions"
            else "First-place votes"
        )
    )


    if "share" in x_column:

        ax.xaxis.set_major_formatter(
            PercentFormatter(
                1.0
            )
        )

    elif (
        x_column == METRICS[0]
    ):

        ax.xaxis.set_major_formatter(
            StrMethodFormatter(
                "${x:,.0f}"
            )
        )

    else:

        ax.xaxis.set_major_formatter(
            StrMethodFormatter(
                "{x:,.0f}"
            )
        )


    ax.yaxis.set_major_formatter(
        StrMethodFormatter(
            "{x:,.0f}"
        )
    )


    pearson_r = (
        plot_data[
            x_column
        ]
        .corr(
            plot_data[
                y_column
            ],
            method="pearson",
        )
    )


    ax.set_title(
        (
            f"{METRIC_LABELS[x_column]} → "
            f"{y_column.replace('_', ' ')}\n"
            f"District: {district} | "
            f"Pearson r = {pearson_r:.2f} | "
            f"R² = {model.rsquared:.2f} | "
            f"n = {len(plot_data)}"
        )
    )


    ax.margins(
        x=0.12,
        y=0.12,
    )

    ax.legend(
        frameon=False
    )

    plt.tight_layout()
    plt.show()


    results = pd.DataFrame(
        {
            "x_metric": [
                x_column
            ],
            "y_metric": [
                y_column
            ],
            "district": [
                district
            ],
            "n": [
                len(plot_data)
            ],
            "slope": [
                model.params[1]
            ],
            "intercept": [
                model.params[0]
            ],
            "pearson_r": [
                pearson_r
            ],
            "r_squared": [
                model.rsquared
            ],
        }
    )


    display(
        results.round(
            3
        )
    )


    return model


## 9. Run the selected regression

Change the configuration at the top and rerun this cell.


In [ ]:
model = plot_simple_regression(
    analysis,
    x_column=X_METRIC,
    y_column=Y_METRIC,
    district=PLOT_DISTRICT,
    show_names=SHOW_NAMES,
)


## Quick notes

- What looks strongest in the race?
- Does the pattern change when I choose another district?
- Does the plot look roughly linear?
- Are there obvious candidates far above or below the line?
- What X should I try next?
